# **Research**

**Done By:**

*   Wiktor Kuranowski | wk149530
*   Conrad Nowowiejski |cn150891
*   Amine Baaza | ab148718



## **1. Topic Justification**

The automotive secondary market is a multi-billion dollar industry where pricing transparency is crucial for both consumers and dealerships. This topic was selected because vehicle depreciation is a complex process influenced by a mix of objective factors (mileage, age) and subjective attributes (color, condition). Understanding the thresholds that classify a vehicle as a "High-Value" asset allows for better inventory management and more accurate financial forecasting in the used car sector.

## **2. Basic Research Question (Hypothesis) and Literature Background**

*   **Research Question:** To what extent do usage-based factors (odometer and age) outperform physical condition and brand prestige in predicting whether a vehicle will sell above the market median?

*   **Hypothesis:** Vehicle usage metrics (odometer reading) have a statistically higher impact on price classification than subjective ratings (condition) or aesthetic features (color).

*   **Literature Background:** Standard economic theories of depreciation suggest that automobiles are "diminishing assets." Literature in this field typically focuses on the Hedonic Pricing Model, which assumes the price of a complex product is determined by the sum of its individual parts and characteristics.






## **3. Database – Source and Description, Descriptive Statistics**

**Source:** Vehicle Sales and Market Trends Dataset from
https://www.kaggle.com/datasets/syedanwarafridi/vehicle-sales-data

**Description:** The dataset contains transaction records including make, model, year, body type, and condition. It provides market benchmarks via Manheim Market Report (MMR) values.

**Descriptive Statistics (Post-Cleaning):**

>  **Sample Size:** ~470,000 observations (exceeding the 3-4k requirement).

 > **Variables:** 11 features including car_age, odometer, condition, and encoded categorical data (make_enc, body_enc).

>**Target Variable:** (is_high_value)

=-=-=-=-=-=-=-=-==-=-=-=-=-=-=-=-=-=-=-=-=-=-=-=-=-=-=-=-=-=-=-=-

# Code

**Loading Data & Basic Stats**

In [0]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler, LabelEncoder
from sklearn.neural_network import MLPClassifier
from sklearn.tree import DecisionTreeClassifier, plot_tree
from sklearn.inspection import permutation_importance
from sklearn.tree import plot_tree
from sklearn.metrics import (
    classification_report, confusion_matrix, accuracy_score,
    roc_curve, auc, precision_recall_curve,
    average_precision_score, f1_score
)

In [0]:
df = pd.read_csv('car_prices.csv')

print(f"Brakujące wartości:\n{df.isnull().sum()}")
print(f"Liczba duplikatów: {df.duplicated().sum()}")
print(f"Typy danych:\n{df.dtypes}")
display(df.head(1))

**Data Cleanig**

In [0]:
df_cleaned = df.dropna()

def detect_outliers_iqr(data, column):
    Q1 = data[column].quantile(0.25)
    Q3 = data[column].quantile(0.75)
    IQR = Q3 - Q1
    lower_bound = Q1 - 1.5 * IQR
    upper_bound = Q3 + 1.5 * IQR
    return lower_bound, upper_bound

price_lb, price_ub = detect_outliers_iqr(df_cleaned, 'sellingprice')
odo_lb, odo_ub = detect_outliers_iqr(df_cleaned, 'odometer')

def remove_outliers(df, column):
    Q1 = df[column].quantile(0.25)
    Q3 = df[column].quantile(0.75)
    IQR = Q3 - Q1
    lower_bound = Q1 - 1.5 * IQR
    upper_bound = Q3 + 1.5 * IQR
    return df[(df[column] >= lower_bound) & (df[column] <= upper_bound)]

df_final = remove_outliers(df_cleaned, 'sellingprice')
df_final = remove_outliers(df_final, 'odometer')
df_final = df_final[df_final['sellingprice'] > 100]

EDA

In [0]:
plt.figure(figsize=(16, 6))
plt.subplot(1, 2, 1)
sns.boxplot(x=df_cleaned['sellingprice'], color='skyblue')
plt.subplot(1, 2, 2)
sns.boxplot(x=df_cleaned['odometer'], color='salmon')
plt.show()

plt.figure(figsize=(12, 6))
sns.countplot(data=df_final, x='make', order=df_final['make'].value_counts().index[:10], palette='viridis')
plt.xticks(rotation=45)
plt.show()


In [0]:

plt.figure(figsize=(10, 8))
correlation_matrix = df_final[['sellingprice', 'odometer', 'mmr', 'condition', 'year']].corr()
sns.heatmap(correlation_matrix, annot=True, cmap='coolwarm', fmt='.2f')
plt.show()


In [0]:

df_final['saledate'] = pd.to_datetime(df_final['saledate'], utc=True)
df_final['sale_month'] = df_final['saledate'].dt.month
df_final.groupby('sale_month')['sellingprice'].mean().plot(marker='o')
plt.show()


In [0]:
sns.regplot(data=df_final.sample(2000), x='condition', y='sellingprice', scatter_kws={'alpha':0.1}, line_kws={'color':'red'})
plt.show()

**Feature Engeneriing**


In [0]:
current_year = df_final['year'].max()
df_final['car_age'] = current_year - df_final['year']
df_final['price_diff'] = df_final['sellingprice'] - df_final['mmr']
df_final['sale_day_of_week'] = df_final['saledate'].dt.day_name()
df_final['is_weekend'] = (df_final['saledate'].dt.dayofweek // 5).astype(int)
df_final['mileage_per_year'] = df_final['odometer'] / (df_final['car_age'] + 1)
df_final['body'] = df_final['body'].str.lower()

median_price = df_final['sellingprice'].median()
df_final['is_high_value'] = (df_final['sellingprice'] > median_price).astype(int)

In [0]:
plt.figure(figsize=(10, 6))
sns.lineplot(data=df_final, x='car_age', y='sellingprice')
plt.title('Depreciation Curve: Average Price by Car Age')
plt.grid(True)
plt.show()

In [0]:
le = LabelEncoder()
categorical_cols = ['make', 'body', 'color', 'state', 'interior']

for col in categorical_cols:
    top_10 = df_final[col].value_counts().index[:10]
    df_final[col] = df_final[col].where(df_final[col].isin(top_10), 'Other')
    df_final[f'{col}_enc'] = le.fit_transform(df_final[col].astype(str))


df_final['transmission_enc'] = (df_final['transmission'] == 'automatic').astype(int)


median_price = df_final['sellingprice'].median()
df_final['is_high_value'] = (df_final['sellingprice'] > median_price).astype(int)

**Model Preparation And tarining**

In [0]:
features = [
    'year', 'condition', 'odometer', 'car_age', 'mileage_per_year',
    'transmission_enc', 'make_enc', 'body_enc', 'color_enc', 'state_enc', 'interior_enc'
]

X = df_final[features]
y = df_final['is_high_value']

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)




## 4. Model I – Neural Network (Theory and Specification)
**Theory:** A Neural Network (Multi-Layer Perceptron) mimics human cognitive processes through layers of interconnected "neurons." It is a non-linear model capable of capturing complex interactions between variables that traditional linear models might miss.

**Specification:**

>**Architecture:** Two hidden layers with 10 and 5 neurons respectively.

>**Activation Function:** ReLU (Rectified Linear Unit).

>**Optimization:** Adam solver with a maximum of 500 iterations.

>**Preprocessing:** Data normalized using StandardScaler.

In [0]:
mlp = MLPClassifier(hidden_layer_sizes=(10, 5), max_iter=500, random_state=42)
mlp.fit(X_train_scaled, y_train)
y_pred_mlp = mlp.predict(X_test_scaled)

## 5. Model II – Decision Tree (Theory and Specification)
**Theory:** A Decision Tree is a non-parametric supervised learning method. It functions by splitting the dataset into branches based on feature values that maximize "information gain" or minimize "Gini impurity." It is highly interpretable, representing a logical flowchart of decisions.

**Specification:**

>**Criterion:** Gini impurity.

>**Max Depth:** 5 (limited to prevent overfitting and ensure clarity for presentation).

>**Features:** Evaluates all 11 variables to find the most significant "split points."

In [0]:
dtree = DecisionTreeClassifier(max_depth=5, random_state=42)
dtree.fit(X_train, y_train)
y_pred_dtree = dtree.predict(X_test)

In [0]:
plt.figure(figsize=(20, 10))
plot_tree(dtree,
          feature_names=features,
          class_names=['Low Value', 'High Value'],
          filled=True,
          fontsize=10,
          max_depth=2)
plt.title("Decision Tree Logic (Top Levels)")
plt.show()

**Models Random Car Test**

In [0]:

sample_car = X_test.iloc[0:1]
sample_car_scaled = X_test_scaled[0:1]

print(f"Car Details:\n{sample_car}")
print(f"Actual Category: {'High Value' if y_test.iloc[0] == 1 else 'Low Value'}")
print(f"NN Prediction: {'High Value' if mlp.predict(sample_car_scaled)[0] == 1 else 'Low Value'}")
print(f"DT Prediction: {'High Value' if dtree.predict(sample_car)[0] == 1 else 'Low Value'}")

**Model Evaluation**

## 6. Results of Estimation and Interpretation
The models were trained on over 470,000 records using 11 variables. The target variable is_high_value was nearly perfectly balanced.

### **Model I: Neural Network (MLP)**
>**Performance:** Achieved an accuracy of 0.8424.

>**Interpretation:** This model shows very high stability, with identical precision and recall for both classes (0.84). It suggests that the neural network successfully captured the non-linear relationship between variables like car_age, odometer, and the final sellingprice.

### **Model II: Decision Tree**
>**Performance:** Achieved an accuracy of 0.8188.

>**Interpretation:** While slightly less accurate than the Neural Network, the Decision Tree shows an interesting bias. It has a higher recall for high-value cars (0.86) compared to the NN, meaning it is better at "catching" almost all expensive cars, even if it occasionally misclassifies low-value ones as expensive (lower precision of 0.79).

In [0]:
def show_metrics(name, y_true, y_pred):
    print(f"\n--- {name} Performance ---")
    print(f"Accuracy: {accuracy_score(y_true, y_pred):.4f}")
    print(classification_report(y_true, y_pred))

show_metrics("Neural Network", y_test, y_pred_mlp)
show_metrics("Decision Tree", y_test, y_pred_dtree)

plt.figure(figsize=(10, 6))
for model, x_data, label, color in [
    (mlp, X_test_scaled, 'Neural Network', 'purple'),
    (dtree, X_test, 'Decision Tree', 'green')
]:
    probs = model.predict_proba(x_data)[:, 1]
    fpr, tpr, _ = roc_curve(y_test, probs)
    plt.plot(fpr, tpr, label=f'{label} (AUC = {auc(fpr, tpr):.3f})', color=color)

plt.plot([0, 1], [0, 1], 'k--')
plt.title('ROC Curve Comparison')
plt.xlabel('False Positive Rate')
plt.ylabel('True Positive Rate')
plt.legend()
plt.show()

## 7. Model Comparison
**Comparison Analysis:** The Neural Network is the more robust "all-around" predictor. However, the Decision Tree is a strong contender because its higher recall for Class 1 makes it a useful tool for sellers who want to ensure they don't overlook potentially high-value inventory.

In [0]:
metrics_summary = pd.DataFrame({
    'Metric': ['Accuracy', 'F1-Score (Macro)', 'Precision (High Value)', 'Recall (High Value)'],
    'Neural Network (MLP)': [
        accuracy_score(y_test, y_pred_mlp),
        f1_score(y_test, y_pred_mlp, average='macro'),
        classification_report(y_test, y_pred_mlp, output_dict=True)['1']['precision'],
        classification_report(y_test, y_pred_mlp, output_dict=True)['1']['recall']
    ],
    'Decision Tree': [
        accuracy_score(y_test, y_pred_dtree),
        f1_score(y_test, y_pred_dtree, average='macro'),
        classification_report(y_test, y_pred_dtree, output_dict=True)['1']['precision'],
        classification_report(y_test, y_pred_dtree, output_dict=True)['1']['recall']
    ]
})

display(metrics_summary.round(4))

## **8. Conclusions Concerning Research Question**
>**Hypothesis Confirmation:** The results strongly support the hypothesis that vehicle attributes like mileage and age can predict market value with over 84% accuracy.

>**Key Driver:** Based on the high performance of both models, the "usage-based" variables (Odometer and Car Age) serve as the most reliable indicators of price category.

>**Business Application:** While the Neural Network offers the best precision, the Decision Tree provides a more actionable "rule-based" system for dealerships due to its ease of interpretation and high recall for premium vehicles.

**Feature Importance**

In [0]:
dt_importance = pd.DataFrame({'Feature': features, 'Importance': dtree.feature_importances_}).sort_values('Importance', ascending=False)


nn_perm = permutation_importance(mlp, X_test_scaled, y_test, n_repeats=5, random_state=42)
nn_importance = pd.DataFrame({'Feature': features, 'Importance': nn_perm.importances_mean}).sort_values('Importance', ascending=False)


In [0]:
fig, ax = plt.subplots(1, 2, figsize=(16, 6))
sns.barplot(x='Importance', y='Feature', data=dt_importance, ax=ax[0], palette='viridis')
ax[0].set_title('Decision Tree Feature Importance')

sns.barplot(x='Importance', y='Feature', data=nn_importance, ax=ax[1], palette='plasma')
ax[1].set_title('Neural Network Feature Importance (Permutation)')
plt.tight_layout()
plt.show()